In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5")

d:\workspace\AI\langchain-academy\intro-2-langchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [4]:
from langchain.agents import  AgentState

class EmailState(AgentState):
    email: str

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem. '
                                                                          "Let's "
                                                                          'meet '
                                                                          'tomorrow '
                                                                          'at '
                                                                          '2 '
                                                                          'PM '
                                                                          'instead. '
                                                                          'Best, '
            

In [10]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, no problem. Let's meet tomorrow at 2 PM instead. Best, Seán."}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, no problem. Let\'s meet tomorrow at 2 PM instead. Best, Seán."}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='da5d85cabf9bf095d3026c70375c4e16')]


In [11]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem. Let's meet tomorrow at 2 PM instead. Best, Seán.


# Approve

In [12]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='bb4dbf94-5a9c-4745-9344-4336d2808e94'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-10T17:56:52.8970385Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4027618800, 'load_duration': 3109954100, 'prompt_eval_count': 338, 'prompt_eval_duration': 128710700, 'eval_count': 72, 'eval_duration': 687951400, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d788a-1664-7793-b1e9-c67dd7f262a8-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '82da85b9-a930-4a39-ba86-306e83646341', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 338, 'output_tokens': 72, 

# Reject

In [15]:
config_2 = {"configurable": {"thread_id": "2"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config_2
)

In [16]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config_2 # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'What '
                                                                          'time '
                                                                          'works '
                                                                          'better '
                                                                          'for '
                                                                          'you '
                 

In [17]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, No problem at all. What time works better for you tomorrow? Best, Your merciful leader, Seán.


In [18]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='bb4dbf94-5a9c-4745-9344-4336d2808e94'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-10T17:56:52.8970385Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4027618800, 'load_duration': 3109954100, 'prompt_eval_count': 338, 'prompt_eval_duration': 128710700, 'eval_count': 72, 'eval_duration': 687951400, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d788a-1664-7793-b1e9-c67dd7f262a8-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '82da85b9-a930-4a39-ba86-306e83646341', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 338, 'output_tokens': 72, 